In [ ]:
    def parse_belief_line(self,type,beliefs):
        for line in beliefs.splitlines():
            line = line.strip()
            if not line:
                return
            
            tokens = line.split()
            if len(tokens) < 3:
                return
            
            tokens = [t.lower() for t in tokens]

            #对手拿了什么
            oppo_hold_first = []
            my_hold_first = []
            my_container_first =  None
            oppo_container_first = None
            if type == "first":
                
                if tokens.count('believe') < 2:
                    return
                first_believe_idx = tokens.index('believe')
                second_believe_idx = tokens.index('believe', first_believe_idx + 1)
                belief_tokens = tokens[second_believe_idx + 1:]
                if len(belief_tokens) < 3:
                    return
                subject = belief_tokens[0]
                predicate = belief_tokens[1]
                obj = belief_tokens[2]

                believer_agent = tokens[2]  # first order 中，第二个 agent 是“相信者”（如 Bob）
                #房间情况
                if "explore" in predicate:
                    room_str,room_name_room_id = self.parse_entity(subject)

                    if obj == 'part':
                        self.oppo_rooms_explored.update({f'{room_str}':'part'})
                    if obj == 'all':
                        self.oppo_rooms_explored.update({f'{room_str}':'all'})
                
                for i, agent_name in enumerate(self.agent_names):
                    if subject.capitalize() == agent_name:
                        agent_id = i
                        if agent_id == self.opponent_agent_id:
                            if 'hold' in predicate:
                                obj_str, name, id = self.parse_entity(obj)
                                type = 0
                                if name.lower() not in self.goal_objects.keys():
                                    oppo_container_first = int(id)
                                    type = 1
                                hold_dic = {'id': int(id), 'type': type, 'name': name, 'contained': [None, None, None], 'contained_name': [None, None, None]}
                                oppo_hold_first.append(hold_dic)
                            elif 'at' in predicate:
                                room_str, name, id = self.parse_entity(obj)
                                oppo_current_room_first = room_str
                        elif agent_id == self.agent_id:
                            if 'hold' in predicate:
                                #判断容器
                                obj_str, name, id = self.parse_entity(obj)
                                type = 0
                                if name.lower() not in self.goal_objects.keys():
                                    my_container_first = int(id)
                                    type = 1
                                hold_dic = {'id': int(id), 'type': type, 'name': name, 'contained': [None, None, None], 'contained_name': [None, None, None]}
                                my_hold_first.append(hold_dic)
                            elif 'at' in predicate:
                                room_str, name, id = self.parse_entity(obj)
                                my_current_room_first = room_str
                #物体判断
                if 'in' in predicate and (subject not in self.agent_names):
                    room_str,room_name,room_id = self.parse_entity(obj)
                    obj_str,obj_name, obj_id = self.parse_entity(subject)
                    if obj_name.lower() in self.goal_objects.keys():
                        self.oppo_object_per_room[room_str][0].append(obj_str)
                    elif 'bed' in obj_name.lower():
                        self.oppo_object_per_room[room_str][2].append(obj_str)
                    else:
                        self.oppo_object_per_room[room_str][1].append(obj_str)

                #第二轮检测


            else:

                oppo_hold_zero = []
                oppo_container_zero = None
                try:
                    believe_idx = tokens.index('believe')  # 不区分大小写
                except ValueError:
                    return
                belief_tokens = tokens[believe_idx + 1:]      # 用原始 tokens 提取内容
                if len(belief_tokens) < 3:
                    return
                subject = belief_tokens[0]
                predicate = belief_tokens[1].upper()         # 谓词统一转大写用于判断
                obj = belief_tokens[2]
                believer_agent = tokens[0]  # zero order 中的主体（如 Alice）

                #房屋
                if "explore" in predicate:
                    room_str,room_name_room_id = self.parse_entity(subject)

                    if obj == 'part':
                        self.rooms_explored.update({f'{room_str}':'part'})
                    if obj == 'all':
                        self.rooms_explored.update({f'{room_str}':'all'})
                #物体
                if 'in' in predicate and (subject not in self.agent_names):
                    room_str,room_name,room_id = self.parse_entity(obj)
                    obj_str,obj_name, obj_id = self.parse_entity(subject)
                    if obj_name.lower() in self.goal_objects.keys():
                        self.object_per_room[room_str][0].append(obj_str)
                    elif 'bed' in obj_name.lower():
                        self.object_per_room[room_str][2].append(obj_str)
                    else:
                        self.object_per_room[room_str][1].append(obj_str)
                #智能体        
                for i, agent_name in enumerate(self.agent_names):
                    if subject.capitalize() == agent_name:
                        if i == self.opponent_agent_id:
                            if 'hold' in predicate:
                                #判断容器
                                obj_str, name, id = self.parse_entity(obj)
                                if obj_name.lower() not in self.goal_objects.keys():
                                    type = 1
                                    oppo_container_zero = int(id)
                                hold_dic = {'id': int(id), 'type': type, 'name': name, 'contained': [None, None, None], 'contained_name': [None, None, None]}
                                oppo_hold_zero.append(hold_dic)
                            elif 'at' in predicate:
                                room_str, name, id = self.parse_entity(obj)
                                my_current_room_first = room_str
                                self.oppo_last_room = room_str


        if type == 'first':
            if oppo_container_first != None or my_container_first != None:
                oppo_contain = []
                my_contain = []
                oppo_contain_name = []
                my_contain_name = []
                for line in beliefs.splitlines():
                    line = line.strip()

                    tokens = line.split()
                    if len(tokens) < 3:
                        return
                    
                    tokens = [t.lower() for t in tokens]

                    if 'contain' in tokens:
                
                        if tokens.count('believe') < 2:
                            return
                        first_believe_idx = tokens.index('believe')
                        second_believe_idx = tokens.index('believe', first_believe_idx + 1)
                        belief_tokens = tokens[second_believe_idx + 1:]
                        if len(belief_tokens) < 3:
                            return
                        subject = belief_tokens[0]
                        predicate = belief_tokens[1]
                        obj = belief_tokens[2]

                        if 'contain' in predicate:
                            con_str,con_name,con_id = self.parse_entity(subject)
                            if oppo_container_first == int(con_id):
                                for index, dic in enumerate(oppo_hold_first):
                                    if dic['id'] == int(con_id):
                                        oppo_contain.append(int(con_id))
                                        oppo_contain_name.append(con_str)
                            if my_container_first == int(con_id):
                                for index, dic in enumerate(my_hold_first):
                                    if dic['id'] == int(con_id):
                                        my_contain.append(int(con_id))
                                        my_contain_name.append(con_str)
                if oppo_container_first:
                    for index,obj in enumerate(oppo_hold_first):
                        if obj['type'] == 1:
                            oppo_hold_first[index]['contained'] = oppo_contain
                            oppo_hold_first[index]['contained_name'] = oppo_contain_name

                if my_container_first:
                    for index,obj in enumerate(my_hold_first):
                        if obj['type'] == 1:
                            my_hold_first[index]['contained'] = my_contain
                            my_hold_first[index]['contained_name'] = my_contain_name
                                


            


        else:
            if oppo_container_zero != None:
                oppo_contain = []
                oppo_contain_name = []
                for line in beliefs.splitlines():
                    line = line.strip()
                    if not line:
                        return
                    
                    tokens = line.split()
                    if len(tokens) < 3:
                        return
                    
                    tokens = [t.lower() for t in tokens]

                    if 'contain' in tokens:
                        try:
                            believe_idx = tokens.index('believe')  # 不区分大小写
                        except ValueError:
                            return
                        belief_tokens = tokens[believe_idx + 1:]      # 用原始 tokens 提取内容
                        if len(belief_tokens) < 3:
                            return
                        subject = belief_tokens[0]
                        predicate = belief_tokens[1].upper()         # 谓词统一转大写用于判断
                        obj = belief_tokens[2]
                        believer_agent = tokens[0]

                        if 'contain' in predicate:
                            con_str,con_name,con_id = self.parse_entity(subject)
                            if oppo_container_zero == int(con_id):
                                for index, dic in enumerate(oppo_hold_zero):
                                    if dic['id'] == int(con_id):
                                        oppo_contain.append(int(con_id))
                                        oppo_contain_name.append(con_str)

                        for index,obj in enumerate(oppo_hold_zero):
                            if obj['type'] == 1:
                                oppo_hold_first[index]['contained'] = oppo_contain
                                oppo_hold_first[index]['contained_name'] = oppo_contain_name